In [1]:
# =====================================================================
# CELL 1: Verify GPU, Install Dependencies, Unzip & Inspect Codebase
# =====================================================================
!pip install -q -U scikit-learn==1.7.1 optuna

import os
import zipfile
import inspect
import json
import torch
from pathlib import Path

# 1. Verify GPU
print("=" * 60)
if torch.cuda.is_available():
    print(f"GPU STATUS: Connected -> {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB VRAM)")
else:
    print("WARNING: GPU not found! Please enable T4 GPU in Runtime -> Change runtime type.")
print("=" * 60)

# 2. Find and Extract Zip File
zip_files = [f for f in os.listdir('.') if f.endswith('.zip')]
print(f"\nFound zip archives in current directory: {zip_files}")

if zip_files:
    # Prefer the zip with 'multi' or 'quantile' in the name, otherwise pick the first zip
    target_zip = next((f for f in zip_files if "multi" in f.lower() or "quantile" in f.lower()), zip_files[0])
    extract_dir = Path("./mh_tft")
    extract_dir.mkdir(exist_ok=True)

    print(f"Extracting '{target_zip}' into {extract_dir}...")
    with zipfile.ZipFile(target_zip, 'r') as zf:
        zf.extractall(extract_dir)

    # Locate the directory containing tft_model.py
    model_paths = list(extract_dir.rglob("tft_model.py"))
    work_dir = model_paths[0].parent if model_paths else extract_dir
    os.chdir(work_dir)

print(f"\nActive Working Directory: {os.getcwd()}")
print("Files in Working Directory:")
for f in sorted(os.listdir('.')):
    if os.path.isfile(f):
        print(f"  - {f:42s} ({os.path.getsize(f) / (1024**2):6.2f} MB)")

# 3. Inspect tft_model.py and dataset.py signatures
print("\n" + "=" * 60)
print("INSPECTING tft_model.py AND dataset.py SIGNATURES:")
print("=" * 60)
if os.path.exists("tft_model.py"):
    import tft_model
    for name, obj in inspect.getmembers(tft_model, inspect.isclass):
        if obj.__module__ == "tft_model":
            print(f"[tft_model.py] class {name}{inspect.signature(obj.__init__)}")
            if hasattr(obj, "forward"):
                print(f"               forward{inspect.signature(obj.forward)}")

if os.path.exists("dataset.py"):
    import dataset
    for name, obj in inspect.getmembers(dataset, inspect.isclass):
        if obj.__module__ == "dataset":
            print(f"[dataset.py]   class {name}{inspect.signature(obj.__init__)}")
    for name, obj in inspect.getmembers(dataset, inspect.isfunction):
        if obj.__module__ == "dataset":
            print(f"[dataset.py]   def {name}{inspect.signature(obj)}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 81.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.4/442.4 kB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 29.9 MB/s eta 0:00:00
GPU STATUS: Connected -> Tesla T4 (14.56 GB VRAM)

Found zip archives in current directory: ['Multi-Horizon Quantile Forecasting.zip']
Extracting 'Multi-Horizon Quantile Forecasting.zip' into mh_tft...

Active Working Directory: /content/mh_tft/Multi-Horizon Quantile Forecasting
Files in Working Directory:
  - dataset.py                                 (  0.01 MB)
  - selected_pairs.csv                         (  0.10 MB)
  - sequence_generation_multivariate_report.json (  0.01 MB)
  - sequence_index_summary.csv                 (  0.07 MB)
  - sequences_test.npz                         (  9.98 MB)
  - sequences_train.npz                        (1924.74 MB)
  - sequences_validation.npz                   ( 10.18 MB)
  - target_sales_scaler.joblib       

In [2]:
# =====================================================================
# CELL 2: Inspect tft_model.py, dataset.py & Sequence Report
# =====================================================================
import json

print("=" * 65)
print("1. CONTENTS OF tft_model.py:")
print("=" * 65)
with open("tft_model.py", "r") as f:
    print(f.read())

print("\n" + "=" * 65)
print("2. CONTENTS OF dataset.py:")
print("=" * 65)
with open("dataset.py", "r") as f:
    print(f.read())

print("\n" + "=" * 65)
print("3. SEQUENCE GENERATION REPORT SUMMARY:")
print("=" * 65)
if os.path.exists("sequence_generation_multivariate_report.json"):
    with open("sequence_generation_multivariate_report.json", "r") as f:
        report = json.load(f)
    print(json.dumps(report, indent=2)[:2000])

1. CONTENTS OF tft_model.py:
import torch
import torch.nn as nn
import torch.nn.functional as F

class GatedResidualNetwork(nn.Module):
    def __init__(self, input_size, hidden_size, output_size=None, dropout=0.1):
        super().__init__()
        output_size = output_size or hidden_size
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, output_size)
        self.gate = nn.Linear(output_size, output_size)
        self.skip = nn.Linear(input_size, output_size) if input_size != output_size else nn.Identity()
        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(output_size)

    def forward(self, x):
        residual = self.skip(x)
        x = F.elu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        gate = torch.sigmoid(self.gate(x))
        return self.norm(gate * x + (1.0 - gate) * residual)

class VariableSelectionNetwork(nn.Module):
    def __init__(self, num_variables, input_size, hidden_size

In [3]:
# =====================================================================
# CELL 3: Fast In-Memory Dataset, Cardinalities & DataLoaders
# =====================================================================
import time
import json
import joblib
import random
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader

# 1. Set Seed & Device
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. Print the remaining part of sequence_generation_multivariate_report.json
with open("sequence_generation_multivariate_report.json", "r") as f:
    full_report = json.load(f)

print("Future Features List:", full_report["sequence_design"]["future_features"])
if "target_scaling" in full_report:
    print("Target Scaling Info:", json.dumps(full_report["target_scaling"], indent=2))

# 3. Load NPZ archives using dataset.py's fast load_npz_mmap
from dataset import load_npz_mmap, STATIC_METADATA_COLS

t0 = time.time()
print("\n1. Memory-mapping uncompressed NPZ archives...")
train_arrays = load_npz_mmap("sequences_train.npz")
val_arrays   = load_npz_mmap("sequences_validation.npz")
test_arrays  = load_npz_mmap("sequences_test.npz")

# 4. Compute Category Mappings (+1 offset so index 0 is reserved for <UNK>)
print("2. Building category mappings (+1 offset for <UNK> safety)...")
cat_categories = {}
cat_cardinalities = []

for col in STATIC_METADATA_COLS:
    uniques = np.unique(train_arrays[col])
    cat_categories[col] = uniques
    cat_cardinalities.append(len(uniques) + 1)
    print(f"   - {col:12s}: {len(uniques):3d} unique -> embedding cardinality {len(uniques) + 1:3d}")

# 5. Fast In-Memory Tensor Dataset (Pre-loads tensors into RAM for ~20s epochs)
class FastInMemoryTFTDataset(Dataset):
    def __init__(self, mmap_arrays, cat_categories, static_cols, split_name="train"):
        t_start = time.time()
        self.past_x = torch.from_numpy(np.array(mmap_arrays["past_x"], dtype=np.float32, copy=True))
        self.future_x = torch.from_numpy(np.array(mmap_arrays["future_x"], dtype=np.float32, copy=True))
        self.y = torch.from_numpy(np.array(mmap_arrays["y"], dtype=np.float32, copy=True))
        self.y_raw = torch.from_numpy(np.array(mmap_arrays["y_raw"], dtype=np.float32, copy=True))

        # Shift codes by +1: unseen categories (-1) become 0 (<UNK>), known become 1..K
        static_matrix = np.column_stack([
            (pd.Categorical(mmap_arrays[col], categories=cat_categories[col]).codes + 1).astype(np.int64)
            for col in static_cols
        ])
        self.static_x = torch.from_numpy(static_matrix)
        print(f"   [{split_name.upper():10s}] Loaded {len(self.y):7,d} sequences into RAM in {time.time() - t_start:.2f}s")

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return {
            "past_x":   self.past_x[idx],
            "future_x": self.future_x[idx],
            "static_x": self.static_x[idx],
            "y":        self.y[idx],
            "y_raw":    self.y_raw[idx],
        }

print("\n3. Pre-loading Train, Validation, and Test Tensors into RAM...")
train_dataset = FastInMemoryTFTDataset(train_arrays, cat_categories, STATIC_METADATA_COLS, "train")
val_dataset   = FastInMemoryTFTDataset(val_arrays,   cat_categories, STATIC_METADATA_COLS, "validation")
test_dataset  = FastInMemoryTFTDataset(test_arrays,  cat_categories, STATIC_METADATA_COLS, "test")

# 6. Create High-Speed DataLoaders (Batch Size = 512)
BATCH_SIZE = 512
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

sample_b = next(iter(train_loader))
print("\n" + "=" * 65)
print(f"DataLoaders Ready in {time.time() - t0:.2f}s | Train Batches (B={BATCH_SIZE}): {len(train_loader):,}")
print(f"  past_x:   {tuple(sample_b['past_x'].shape)}")
print(f"  future_x: {tuple(sample_b['future_x'].shape)}")
print(f"  static_x: {tuple(sample_b['static_x'].shape)}")
print(f"  y:        {tuple(sample_b['y'].shape)} | min={sample_b['y'].min():.2f}, max={sample_b['y'].max():.2f}")
print(f"  y_raw:    {tuple(sample_b['y_raw'].shape)} | min={sample_b['y_raw'].min():.1f}, max={sample_b['y_raw'].max():.1f}")
print("=" * 65)

Future Features List: ['day_of_week_sin', 'day_of_week_cos', 'month_sin', 'month_cos', 'day_of_month_sin', 'day_of_month_cos', 'week_of_year_sin', 'week_of_year_cos', 'time_idx', 'is_holiday', 'is_event', 'is_additional', 'is_work_day', 'is_transfer']

1. Memory-mapping uncompressed NPZ archives...
2. Building category mappings (+1 offset for <UNK> safety)...
   - store_nbr   :  53 unique -> embedding cardinality  54
   - item_nbr    : 122 unique -> embedding cardinality 123
   - family      :  32 unique -> embedding cardinality  33
   - class       :  65 unique -> embedding cardinality  66
   - perishable  :   2 unique -> embedding cardinality   3
   - city        :  22 unique -> embedding cardinality  23
   - state       :  16 unique -> embedding cardinality  17
   - type        :   5 unique -> embedding cardinality   6
   - cluster     :  17 unique -> embedding cardinality  18

3. Pre-loading Train, Validation, and Test Tensors into RAM...
   [TRAIN     ] Loaded 624,451 sequences in

In [4]:
# =====================================================================
# CELL 4: Scale-Aware Quantile/Raw Mixed Loss & Inverse Scaler
# =====================================================================
import joblib
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import r2_score

# 1. Load Scaler parameters from target_sales_scaler.joblib
scaler_artifact = joblib.load("target_sales_scaler.joblib")
target_scaler = scaler_artifact["target_scaler"]

SCALER_MEAN = float(target_scaler.mean_[0])
SCALER_SCALE = float(target_scaler.scale_[0])
print(f"Loaded Scaler Params -> Mean: {SCALER_MEAN:.4f}, Scale (Std): {SCALER_SCALE:.4f}")

# 2. Pinball (Quantile) Loss for P10, P50, P90
class QuantileLoss(nn.Module):
    def __init__(self, quantiles=[0.1, 0.5, 0.9]):
        super().__init__()
        self.quantiles = quantiles

    def forward(self, preds, targets):
        # preds: [B, 7, 3], targets: [B, 7]
        targets_expanded = targets.unsqueeze(-1) # [B, 7, 1]
        losses = []
        for i, q in enumerate(self.quantiles):
            error = targets_expanded - preds[:, :, i:i+1]
            loss = torch.max((q - 1.0) * error, q * error)
            losses.append(loss)
        return torch.mean(torch.cat(losses, dim=-1))

# 3. Scale-Aware Mixed Loss (Penalizes raw sales error directly)
class ScaleAwareTFTLoss(nn.Module):
    def __init__(self, mean, scale, quantiles=[0.1, 0.5, 0.9], raw_weight=0.30):
        super().__init__()
        self.quantile_loss = QuantileLoss(quantiles)
        self.mean = mean
        self.scale = scale
        self.raw_weight = raw_weight

    def forward(self, preds, targets_scaled, targets_raw):
        # 1. Loss on scaled log-space quantiles
        loss_log = self.quantile_loss(preds, targets_scaled)

        # 2. Point prediction is P50 (median, index 1)
        p50 = preds[:, :, 1]

        # 3. Differentiable inverse scaling to raw units
        unscaled_log = p50 * self.scale + self.mean
        preds_raw = torch.expm1(torch.clamp(unscaled_log, min=-5.0, max=14.0))

        # 4. Normalized raw-space error (penalizes misses on large volume items)
        scale_norm = torch.mean(targets_raw) + 1.0
        raw_mae = torch.mean(torch.abs(preds_raw - targets_raw)) / scale_norm
        raw_rmse = torch.sqrt(torch.mean((preds_raw - targets_raw) ** 2) + 1e-6) / scale_norm
        loss_raw = 0.5 * raw_mae + 0.5 * raw_rmse

        return (1.0 - self.raw_weight) * loss_log + self.raw_weight * loss_raw

criterion = ScaleAwareTFTLoss(SCALER_MEAN, SCALER_SCALE, quantiles=[0.1, 0.5, 0.9], raw_weight=0.30)

# 4. Inverse Scaler with Smearing / Multiplier Correction
def inverse_scale(predictions, scaler, bias_multiplier=1.0):
    if isinstance(predictions, torch.Tensor):
        preds_np = predictions.detach().cpu().numpy()
    else:
        preds_np = np.asarray(predictions)

    orig_shape = preds_np.shape
    flattened = preds_np.reshape(-1, 1)
    unscaled_log = scaler.inverse_transform(flattened)
    raw_sales = np.expm1(unscaled_log).reshape(orig_shape) * bias_multiplier
    return np.clip(raw_sales, a_min=0.0, a_max=None)

# 5. Metric Computation
def compute_metrics(y_true_raw, y_pred_raw):
    abs_errors = np.abs(y_true_raw - y_pred_raw)
    mae = np.mean(abs_errors)
    rmse = np.sqrt(np.mean((y_true_raw - y_pred_raw) ** 2))
    wape = (np.sum(abs_errors) / (np.sum(y_true_raw) + 1e-8)) * 100.0
    r2 = r2_score(y_true_raw.ravel(), y_pred_raw.ravel())
    return {
        "MAE": round(float(mae), 4),
        "RMSE": round(float(rmse), 4),
        "WAPE (%)": round(float(wape), 2),
        "R2": round(float(r2), 4)
    }

print("STATUS: Scale-Aware Loss & Metrics initialized successfully!")

Loaded Scaler Params -> Mean: 3.2521, Scale (Std): 1.2694
STATUS: Scale-Aware Loss & Metrics initialized successfully!


In [5]:
# =====================================================================
# CELL 5: Instantiate High-Capacity TFT & Dry-Run Sanity Check
# =====================================================================
import time
import torch
from tft_model import TemporalFusionTransformer

print("=" * 65)
print("BUILDING HIGH-CAPACITY TEMPORAL FUSION TRANSFORMER (TFT)")
print("=" * 65)

# 1. Instantiate model with 128 hidden units and 3 quantiles (P10, P50, P90)
model_config = {
    "static_cardinalities": cat_cardinalities,
    "past_features": 18,
    "future_features": 14,
    "hidden_size": 128,
    "embedding_size": 32,
    "num_heads": 4,
    "dropout": 0.10,
    "forecast_horizon": 7,
    "num_quantiles": 3
}

model = TemporalFusionTransformer(**model_config).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Device:                   {DEVICE} ({torch.cuda.get_device_name(0)})")
print(f"Total Trainable Params:   {total_params:,} (vs ~228K previously)")
print(f"Hidden Size:              {model_config['hidden_size']}")
print(f"Num Heads:                {model_config['num_heads']}")
print(f"Forecast Horizon:         {model_config['forecast_horizon']} days")
print(f"Quantiles:                {model_config['num_quantiles']} (P10, P50, P90)")

# 2. Test forward pass with mixed precision (AMP)
t0 = time.time()
sample_b = next(iter(train_loader))
past_x   = sample_b["past_x"].to(DEVICE)
future_x = sample_b["future_x"].to(DEVICE)
static_x = sample_b["static_x"].to(DEVICE)
targets  = sample_b["y"].to(DEVICE)
y_raw    = sample_b["y_raw"].to(DEVICE)

with torch.amp.autocast('cuda'):
    sample_preds = model(past_x, future_x, static_x)
    sample_loss  = criterion(sample_preds, targets, y_raw)

# 3. Test backward step
optimizer_test = torch.optim.AdamW(model.parameters(), lr=1e-3)
scaler_test = torch.amp.GradScaler('cuda')
scaler_test.scale(sample_loss).backward()
scaler_test.step(optimizer_test)
scaler_test.update()
optimizer_test.zero_grad()

print("\n" + "=" * 65)
print("SANITY CHECK RESULTS:")
print("=" * 65)
print(f"Output Shape:             {tuple(sample_preds.shape)} (Expected: (512, 7, 3))")
print(f"Initial Step Loss:        {sample_loss.item():.4f}")
print(f"Contains NaNs:            {torch.isnan(sample_preds).any().item()}")
print(f"Step Time:                {time.time() - t0:.2f}s")
print(f"Allocated GPU VRAM:       {torch.cuda.memory_allocated() / (1024**2):.1f} MB / {torch.cuda.get_device_properties(0).total_memory / (1024**2):.1f} MB")
print("STATUS: Model architecture, AMP, and mixed loss verified successfully!")
print("=" * 65)

BUILDING HIGH-CAPACITY TEMPORAL FUSION TRANSFORMER (TFT)
Device:                   cuda (Tesla T4)
Total Trainable Params:   3,104,553 (vs ~228K previously)
Hidden Size:              128
Num Heads:                4
Forecast Horizon:         7 days
Quantiles:                3 (P10, P50, P90)

SANITY CHECK RESULTS:
Output Shape:             (512, 7, 3) (Expected: (512, 7, 3))
Initial Step Loss:        0.6977
Contains NaNs:            False
Step Time:                7.33s
Allocated GPU VRAM:       54.1 MB / 14912.7 MB
STATUS: Model architecture, AMP, and mixed loss verified successfully!


In [6]:
# =====================================================================
# CELL 6: High-Capacity TFT Training Loop (AMP + OneCycleLR)
# =====================================================================
import time
import copy
import torch

EPOCHS = 15
best_val_rmse = float("inf")
best_model_path = "tft_upgraded_champion.pth"

# 1. Optimizer, AMP Scaler, and Learning Rate Scheduler
optimizer = torch.optim.AdamW(model.parameters(), lr=1.5e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=2.5e-3,
    epochs=EPOCHS,
    steps_per_epoch=len(train_loader),
    pct_start=0.15,
    div_factor=10.0,
    final_div_factor=1e3
)
scaler_amp = torch.amp.GradScaler('cuda')

# 2. Train One Epoch Function
def train_one_epoch(model, loader, optimizer, scheduler, scaler_amp):
    model.train()
    running_loss = 0.0

    for batch in loader:
        past_x   = batch["past_x"].to(DEVICE, non_blocking=True)
        future_x = batch["future_x"].to(DEVICE, non_blocking=True)
        static_x = batch["static_x"].to(DEVICE, non_blocking=True)
        targets  = batch["y"].to(DEVICE, non_blocking=True)
        y_raw    = batch["y_raw"].to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast('cuda'):
            preds = model(past_x, future_x, static_x)
            loss = criterion(preds, targets, y_raw)

        scaler_amp.scale(loss).backward()
        scaler_amp.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler_amp.step(optimizer)
        scaler_amp.update()
        scheduler.step()

        running_loss += loss.item()

    return running_loss / len(loader)

# 3. Validation Evaluation Function (Evaluates P50 Median Predictions)
@torch.no_grad()
def evaluate_validation(model, loader):
    model.eval()
    running_loss = 0.0
    all_p50_preds = []
    all_trues_raw = []

    for batch in loader:
        past_x   = batch["past_x"].to(DEVICE, non_blocking=True)
        future_x = batch["future_x"].to(DEVICE, non_blocking=True)
        static_x = batch["static_x"].to(DEVICE, non_blocking=True)
        targets  = batch["y"].to(DEVICE, non_blocking=True)
        y_raw    = batch["y_raw"].to(DEVICE, non_blocking=True)

        with torch.amp.autocast('cuda'):
            preds = model(past_x, future_x, static_x)
            loss = criterion(preds, targets, y_raw)

        running_loss += loss.item()
        # preds: [B, 7, 3] -> P50 is index 1
        all_p50_preds.append(preds[:, :, 1].float().cpu())
        all_trues_raw.append(batch["y_raw"])

    avg_loss = running_loss / len(loader)
    concat_p50 = torch.cat(all_p50_preds, dim=0).numpy()
    concat_trues_raw = torch.cat(all_trues_raw, dim=0).numpy()

    # Invert log-space predictions back to raw units
    raw_p50_preds = inverse_scale(concat_p50, target_scaler, bias_multiplier=1.0)
    metrics = compute_metrics(concat_trues_raw, raw_p50_preds)
    metrics["Loss"] = round(avg_loss, 4)

    return metrics, concat_p50, concat_trues_raw

# 4. Execute Full Training Loop
print("=" * 70)
print(f"TRAINING 3.1M-PARAM TFT FOR {EPOCHS} EPOCHS ({len(train_loader)} BATCHES/EPOCH)")
print("=" * 70)

training_history = []
start_all_time = time.time()

for epoch in range(1, EPOCHS + 1):
    epoch_t0 = time.time()

    train_loss = train_one_epoch(model, train_loader, optimizer, scheduler, scaler_amp)
    val_metrics, _, _ = evaluate_validation(model, val_loader)

    epoch_time = time.time() - epoch_t0
    current_lr = optimizer.param_groups[0]["lr"]

    training_history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_metrics["Loss"],
        "val_mae": val_metrics["MAE"],
        "val_rmse": val_metrics["RMSE"],
        "val_wape": val_metrics["WAPE (%)"],
        "val_r2": val_metrics["R2"],
        "lr": current_lr,
        "time": epoch_time
    })

    print(f"Epoch [{epoch:02d}/{EPOCHS:02d}] | Train Loss: {train_loss:.4f} | Val Loss: {val_metrics['Loss']:.4f} | "
          f"MAE: {val_metrics['MAE']:6.3f} | RMSE: {val_metrics['RMSE']:6.3f} | "
          f"WAPE: {val_metrics['WAPE (%)']:5.2f}% | R²: {val_metrics['R2']:.4f} | "
          f"LR: {current_lr:.2e} | Time: {epoch_time:4.1f}s")

    # Checkpoint best weights according to Validation RMSE
    if val_metrics["RMSE"] < best_val_rmse:
        improvement = best_val_rmse - val_metrics["RMSE"]
        best_val_rmse = val_metrics["RMSE"]
        torch.save(model.state_dict(), best_model_path)
        print(f"  >>> Checkpoint Saved! Best Val RMSE dropped to {best_val_rmse:.3f} (-{improvement:.3f})")

total_train_time = time.time() - start_all_time
print("\n" + "=" * 70)
print(f"TRAINING COMPLETE IN {total_train_time / 60:.2f} MINUTES!")
print(f"Best Validation RMSE Achieved: {best_val_rmse:.3f}")
print("=" * 70)

TRAINING 3.1M-PARAM TFT FOR 15 EPOCHS (1220 BATCHES/EPOCH)
Epoch [01/15] | Train Loss: 0.2350 | Val Loss: 0.2172 | MAE: 13.226 | RMSE: 38.570 | WAPE: 30.98% | R²: 0.6970 | LR: 1.18e-03 | Time: 150.6s
  >>> Checkpoint Saved! Best Val RMSE dropped to 38.570 (-inf)
Epoch [02/15] | Train Loss: 0.2063 | Val Loss: 0.2129 | MAE: 13.276 | RMSE: 37.754 | WAPE: 31.09% | R²: 0.7097 | LR: 2.43e-03 | Time: 152.2s
  >>> Checkpoint Saved! Best Val RMSE dropped to 37.754 (-0.817)
Epoch [03/15] | Train Loss: 0.1973 | Val Loss: 0.1980 | MAE: 12.164 | RMSE: 34.250 | WAPE: 28.49% | R²: 0.7611 | LR: 2.48e-03 | Time: 150.4s
  >>> Checkpoint Saved! Best Val RMSE dropped to 34.250 (-3.504)
Epoch [04/15] | Train Loss: 0.1910 | Val Loss: 0.1977 | MAE: 12.167 | RMSE: 34.895 | WAPE: 28.50% | R²: 0.7520 | LR: 2.39e-03 | Time: 158.8s
Epoch [05/15] | Train Loss: 0.1894 | Val Loss: 0.1973 | MAE: 12.076 | RMSE: 34.836 | WAPE: 28.28% | R²: 0.7528 | LR: 2.22e-03 | Time: 161.3s
Epoch [06/15] | Train Loss: 0.1851 | Val Lo

In [7]:
# =====================================================================
# CELL 7: Smearing Calibration, Final Test Evaluation & CSV Export
# =====================================================================
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import r2_score

print("=" * 65)
print("LOADING BEST CHECKPOINT & CALIBRATING PREDICTIONS")
print("=" * 65)

# 1. Load best checkpoint
model.load_state_dict(torch.load("tft_upgraded_champion.pth", map_location=DEVICE))
model.eval()

# 2. Extract Validation P50 predictions and ground truth
_, val_p50_scaled, val_trues_raw = evaluate_validation(model, val_loader)

# 3. Grid search for optimal smearing / variance-correction multiplier
best_mult = 1.0
best_tuned_rmse = float("inf")

# Search multipliers between 0.90 and 1.20
for mult in np.linspace(0.90, 1.20, 121):
    cand_preds = inverse_scale(val_p50_scaled, target_scaler, bias_multiplier=mult)
    cand_rmse = np.sqrt(np.mean((val_trues_raw - cand_preds) ** 2))
    if cand_rmse < best_tuned_rmse:
        best_tuned_rmse = cand_rmse
        best_mult = mult

val_uncal_rmse = np.sqrt(np.mean((val_trues_raw - inverse_scale(val_p50_scaled, target_scaler, 1.0)) ** 2))
print(f"Validation Raw RMSE (mult=1.000):   {val_uncal_rmse:.3f}")
print(f"Optimal Smearing Multiplier:        {best_mult:.4f}")
print(f"Validation Calibrated RMSE:         {best_tuned_rmse:.3f} (Improvement: {val_uncal_rmse - best_tuned_rmse:+.3f})")

# 4. Evaluate on Unseen Test Set
test_metrics_uncal, test_p50_scaled, test_trues_raw = evaluate_validation(model, test_loader)
test_preds_cal = inverse_scale(test_p50_scaled, target_scaler, bias_multiplier=best_mult)
test_metrics_cal = compute_metrics(test_trues_raw, test_preds_cal)

print("\n" + "=" * 65)
print("FINAL TEST SET PERFORMANCE (CHAMPION COMPARISON):")
print("=" * 65)
print(f"Baseline Notebook (Cell 8/9):")
print(f"  - Test MAE   : 12.387 units")
print(f"  - Test RMSE  : 29.543 units")
print(f"  - Test WAPE  : 27.99%")
print(f"  - Test R²    : 0.8189")
print("-" * 65)
print(f"Upgraded 3.1M TFT (Raw mult=1.000):")
print(f"  - Test MAE   : {test_metrics_uncal['MAE']:.3f} units")
print(f"  - Test RMSE  : {test_metrics_uncal['RMSE']:.3f} units")
print(f"  - Test WAPE  : {test_metrics_uncal['WAPE (%)']:.2f}%")
print(f"  - Test R²    : {test_metrics_uncal['R2']:.4f}")
print("-" * 65)
print(f"Upgraded 3.1M TFT (Calibrated mult={best_mult:.4f}):")
print(f"  - Test MAE   : {test_metrics_cal['MAE']:.3f} units")
print(f"  - Test RMSE  : {test_metrics_cal['RMSE']:.3f} units")
print(f"  - Test WAPE  : {test_metrics_cal['WAPE (%)']:.2f}%")
print(f"  - Test R²    : {test_metrics_cal['R2']:.4f}")
print("=" * 65)

# 5. Day-by-Day Forecast Accuracy Across 7-Day Horizon
day_metrics = []
for d in range(7):
    day_true = test_trues_raw[:, d]
    day_pred = test_preds_cal[:, d]
    d_mae = np.mean(np.abs(day_true - day_pred))
    d_rmse = np.sqrt(np.mean((day_true - day_pred) ** 2))
    d_wape = (np.sum(np.abs(day_true - day_pred)) / (np.sum(day_true) + 1e-8)) * 100.0
    d_r2 = r2_score(day_true, day_pred)
    day_metrics.append({
        "Lead Time": f"Day +{d+1}",
        "MAE": round(float(d_mae), 3),
        "RMSE": round(float(d_rmse), 3),
        "WAPE (%)": round(float(d_wape), 2),
        "R² Score": round(float(d_r2), 4)
    })

df_days = pd.DataFrame(day_metrics)
print("\nDAY-BY-DAY TEST SET ACCURACY:")
print(df_days.to_string(index=False))

# 6. Export Test Forecasts to CSV
test_archive = np.load("sequences_test.npz", mmap_mode="r")
export_df = pd.DataFrame({
    "store_nbr": test_archive["store_nbr"],
    "item_nbr": test_archive["item_nbr"],
    "target_start_date": test_archive["target_start_date"],
    "target_end_date": test_archive["target_end_date"]
})

for d in range(7):
    export_df[f"actual_day_{d+1}"] = test_trues_raw[:, d]
    export_df[f"pred_day_{d+1}"] = test_preds_cal[:, d]

export_df.to_csv("tft_test_forecasts_upgraded.csv", index=False)
print(f"\nSaved {len(export_df):,} test forecasts to 'tft_test_forecasts_upgraded.csv'!")

# 7. Cross-reference with selected_pairs.csv if present
if os.path.exists("selected_pairs.csv"):
    pairs_df = pd.read_csv("selected_pairs.csv")
    print(f"Loaded selected_pairs.csv: {len(pairs_df)} store-item pairs.")

LOADING BEST CHECKPOINT & CALIBRATING PREDICTIONS
Validation Raw RMSE (mult=1.000):   34.250
Optimal Smearing Multiplier:        1.0000
Validation Calibrated RMSE:         34.250 (Improvement: -0.000)

FINAL TEST SET PERFORMANCE (CHAMPION COMPARISON):
Baseline Notebook (Cell 8/9):
  - Test MAE   : 12.387 units
  - Test RMSE  : 29.543 units
  - Test WAPE  : 27.99%
  - Test R²    : 0.8189
-----------------------------------------------------------------
Upgraded 3.1M TFT (Raw mult=1.000):
  - Test MAE   : 12.654 units
  - Test RMSE  : 28.036 units
  - Test WAPE  : 28.59%
  - Test R²    : 0.8369
-----------------------------------------------------------------
Upgraded 3.1M TFT (Calibrated mult=1.0000):
  - Test MAE   : 12.654 units
  - Test RMSE  : 28.036 units
  - Test WAPE  : 28.59%
  - Test R²    : 0.8369

DAY-BY-DAY TEST SET ACCURACY:
Lead Time    MAE   RMSE  WAPE (%)  R² Score
   Day +1 12.487 30.892     26.88    0.8229
   Day +2 12.283 29.667     27.33    0.8267
   Day +3 12.612 29

In [8]:
# =====================================================================
# CELL 8: Multi-Quantile Blending (P10, P50, P90) & Horizon Calibration
# =====================================================================
import numpy as np
import pandas as pd
import torch
from scipy.optimize import minimize

# 1. Helper to extract ALL 3 quantiles [N, 7, 3] in raw sales units
@torch.no_grad()
def get_all_quantiles_raw(model, loader):
    model.eval()
    all_preds = []
    all_trues = []
    for batch in loader:
        past_x   = batch["past_x"].to(DEVICE, non_blocking=True)
        future_x = batch["future_x"].to(DEVICE, non_blocking=True)
        static_x = batch["static_x"].to(DEVICE, non_blocking=True)
        with torch.amp.autocast('cuda'):
            preds = model(past_x, future_x, static_x) # [B, 7, 3]
        all_preds.append(preds.float().cpu())
        all_trues.append(batch["y_raw"])

    preds_scaled = torch.cat(all_preds, dim=0).numpy() # [N, 7, 3]
    trues_raw = torch.cat(all_trues, dim=0).numpy()    # [N, 7]

    # Inverse scale each quantile (P10, P50, P90) to raw units
    q_raw = np.zeros_like(preds_scaled)
    for q_idx in range(3):
        q_raw[:, :, q_idx] = inverse_scale(preds_scaled[:, :, q_idx], target_scaler, 1.0)
    return preds_scaled, q_raw, trues_raw

# Load the best checkpoint
model.load_state_dict(torch.load("tft_upgraded_champion.pth", map_location=DEVICE))
val_scaled, val_q_raw, val_trues = get_all_quantiles_raw(model, val_loader)
test_scaled, test_q_raw, test_trues = get_all_quantiles_raw(model, test_loader)

# 2. Optimize Quantile Blending Weights per Horizon Day on Validation Set
# Blending P10, P50, P90 in log-space and raw-space to jointly minimize MAE & RMSE
optimal_test_preds = np.zeros_like(test_trues)
optimal_val_preds  = np.zeros_like(val_trues)

print("Optimizing Quantile Blend (P10, P50, P90) per Horizon Day on Validation Set...")
for d in range(7):
    # Features: [P10_log, P50_log, P90_log] on day d
    X_val_log  = val_scaled[:, d, :]
    X_test_log = test_scaled[:, d, :]
    y_val_d    = val_trues[:, d]

    def loss_func(weights):
        w1, w2, w3, bias = weights
        blended_log = w1 * X_val_log[:, 0] + w2 * X_val_log[:, 1] + w3 * X_val_log[:, 2] + bias
        pred_raw = inverse_scale(blended_log, target_scaler, 1.0)
        mae = np.mean(np.abs(y_val_d - pred_raw))
        rmse = np.sqrt(np.mean((y_val_d - pred_raw) ** 2))
        return 0.6 * mae + 0.4 * rmse

    res = minimize(loss_func, [0.05, 0.90, 0.05, 0.0], method="Nelder-Mead")
    w1, w2, w3, b = res.x

    val_blend_log  = w1 * X_val_log[:, 0]  + w2 * X_val_log[:, 1]  + w3 * X_val_log[:, 2]  + b
    test_blend_log = w1 * X_test_log[:, 0] + w2 * X_test_log[:, 1] + w3 * X_test_log[:, 2] + b

    optimal_val_preds[:, d]  = inverse_scale(val_blend_log, target_scaler, 1.0)
    optimal_test_preds[:, d] = inverse_scale(test_blend_log, target_scaler, 1.0)
    print(f"  Day +{d+1} Weights -> P10: {w1:+.3f}, P50: {w2:+.3f}, P90: {w3:+.3f}, Shift: {b:+.4f}")

val_blend_metrics  = compute_metrics(val_trues, optimal_val_preds)
test_blend_metrics = compute_metrics(test_trues, optimal_test_preds)

print("\n" + "=" * 65)
print("TEST SET RESULTS AFTER QUANTILE BLENDING (P10 + P50 + P90):")
print("=" * 65)
print(f"Baseline Notebook        -> MAE: 12.387 | RMSE: 29.543 | WAPE: 27.99% | R²: 0.8189")
print(f"Upgraded TFT (P50 only)  -> MAE: {test_metrics_uncal['MAE']:6.3f} | RMSE: {test_metrics_uncal['RMSE']:6.3f} | WAPE: {test_metrics_uncal['WAPE (%)']:5.2f}% | R²: {test_metrics_uncal['R2']:.4f}")
print(f"Upgraded TFT (Q-Blended) -> MAE: {test_blend_metrics['MAE']:6.3f} | RMSE: {test_blend_metrics['RMSE']:6.3f} | WAPE: {test_blend_metrics['WAPE (%)']:5.2f}% | R²: {test_blend_metrics['R2']:.4f}")
print("=" * 65)

Optimizing Quantile Blend (P10, P50, P90) per Horizon Day on Validation Set...
  Day +1 Weights -> P10: +0.172, P50: +0.665, P90: +0.186, Shift: -0.0293
  Day +2 Weights -> P10: +0.151, P50: +0.809, P90: +0.052, Shift: +0.0189
  Day +3 Weights -> P10: +0.115, P50: +0.815, P90: +0.085, Shift: -0.0061
  Day +4 Weights -> P10: +0.083, P50: +0.758, P90: +0.175, Shift: -0.0427
  Day +5 Weights -> P10: +0.110, P50: +0.814, P90: +0.085, Shift: -0.0115
  Day +6 Weights -> P10: +0.060, P50: +0.799, P90: +0.163, Shift: -0.0734
  Day +7 Weights -> P10: +0.085, P50: +0.697, P90: +0.246, Shift: -0.0888

TEST SET RESULTS AFTER QUANTILE BLENDING (P10 + P50 + P90):
Baseline Notebook        -> MAE: 12.387 | RMSE: 29.543 | WAPE: 27.99% | R²: 0.8189
Upgraded TFT (P50 only)  -> MAE: 12.654 | RMSE: 28.036 | WAPE: 28.59% | R²: 0.8369
Upgraded TFT (Q-Blended) -> MAE: 12.489 | RMSE: 28.120 | WAPE: 28.22% | R²: 0.8359


In [10]:
# =====================================================================
# CELL 9: Enhanced TFT (Full 35-Day Attention + AR Sales Highway)
# =====================================================================
import time
import copy
import torch
import torch.nn as nn
import torch.nn.functional as F
from tft_model import GatedResidualNetwork, StaticVariableSelectionNetwork, VariableSelectionNetwork

class EnhancedTemporalFusionTransformer(nn.Module):
    def __init__(
        self,
        static_cardinalities,
        past_features=18,
        future_features=14,
        hidden_size=96,
        embedding_size=24,
        num_heads=4,
        dropout=0.20,
        forecast_horizon=7,
        num_quantiles=3
    ):
        super().__init__()
        self.hidden_size = hidden_size
        self.forecast_horizon = forecast_horizon
        self.num_quantiles = num_quantiles

        # 1. Static Categorical Embeddings & Selection
        self.static_embeddings = nn.ModuleList([
            nn.Embedding(cardinality, embedding_size)
            for cardinality in static_cardinalities
        ])
        self.static_vsn = StaticVariableSelectionNetwork(
            len(static_cardinalities), embedding_size, hidden_size, dropout
        )

        # 2. Temporal Variable Encoders & VSNs
        self.past_variable_embeddings = nn.ModuleList([
            nn.Linear(1, hidden_size) for _ in range(past_features)
        ])
        self.future_variable_embeddings = nn.ModuleList([
            nn.Linear(1, hidden_size) for _ in range(future_features)
        ])
        self.past_vsn = VariableSelectionNetwork(past_features, hidden_size, hidden_size, dropout)
        self.future_vsn = VariableSelectionNetwork(future_features, hidden_size, hidden_size, dropout)

        # 3. 2-Layer Sequence Encoder-Decoder LSTM with Gated Skip Connection
        self.encoder_lstm = nn.LSTM(hidden_size, hidden_size, num_layers=2, dropout=dropout, batch_first=True)
        self.decoder_lstm = nn.LSTM(hidden_size, hidden_size, num_layers=2, dropout=dropout, batch_first=True)
        self.lstm_Merge_norm = nn.LayerNorm(hidden_size)

        # 4. Static Context Enrichment across ALL 35 timesteps (28 Past + 7 Future)
        self.static_enrichment = GatedResidualNetwork(hidden_size * 2, hidden_size, hidden_size, dropout)

        # 5. Full 35-Day Multi-Head Attention (Decoder attends to all 28 Past + 7 Future days!)
        self.attention = nn.MultiheadAttention(
            embed_dim=hidden_size, num_heads=num_heads, dropout=dropout, batch_first=True
        )
        self.attention_norm = nn.LayerNorm(hidden_size)
        self.final_grn = GatedResidualNetwork(hidden_size * 2, hidden_size, hidden_size, dropout)

        # 6. Direct Autoregressive (AR) Highway from Past 28-Day Sales & Promotions
        # past_x[:, :, 0] is unit_sales_scaled, past_x[:, :, 1] is onpromotion
        self.ar_sales_linear = nn.Linear(28, forecast_horizon * num_quantiles)
        self.ar_sales_grn = GatedResidualNetwork(28 * 2, hidden_size, forecast_horizon * num_quantiles, dropout)

        # 7. Multi-Horizon Quantile Projection Head
        self.forecast_head = nn.Linear(hidden_size, num_quantiles)

    def forward(self, past_x, future_x, static_x):
        B = past_x.size(0)

        # 1. Static Context
        static_emb = torch.stack([
            emb(static_x[:, i]) for i, emb in enumerate(self.static_embeddings)
        ], dim=1)
        static_context, _ = self.static_vsn(static_emb) # [B, H]

        # 2. Past & Future Variable Selection
        past_emb = torch.stack([
            emb(past_x[:, :, i:i+1]) for i, emb in enumerate(self.past_variable_embeddings)
        ], dim=2)
        past_selected, _ = self.past_vsn(past_emb) # [B, 28, H]

        future_emb = torch.stack([
            emb(future_x[:, :, i:i+1]) for i, emb in enumerate(self.future_variable_embeddings)
        ], dim=2)
        future_selected, _ = self.future_vsn(future_emb) # [B, 7, H]

        # 3. LSTM Encoding & Decoding with Residual Skip
        enc_out, (h, c) = self.encoder_lstm(past_selected)
        dec_out, _ = self.decoder_lstm(future_selected, (h, c))

        enc_seq = self.lstm_Merge_norm(enc_out + past_selected)    # [B, 28, H]
        dec_seq = self.lstm_Merge_norm(dec_out + future_selected)  # [B, 7, H]

        # 4. Concatenate Full 35-Day Sequence for Static Enrichment & Cross/Self-Attention
        full_seq = torch.cat([enc_seq, dec_seq], dim=1) # [B, 35, H]
        static_exp = static_context.unsqueeze(1).expand(-1, full_seq.size(1), -1)
        enriched_full = self.static_enrichment(torch.cat([full_seq, static_exp], dim=-1)) # [B, 35, H]

        # 5. Future 7 days attend over the ENTIRE 35-day enriched sequence (Past 28 + Future 7)
        queries_future = enriched_full[:, -self.forecast_horizon:, :] # [B, 7, H]
        attn_out, _ = self.attention(queries_future, enriched_full, enriched_full)
        attn_res = self.attention_norm(attn_out + queries_future)

        # Combine with decoder LSTM output before final GRN
        final_out = self.final_grn(torch.cat([attn_res, dec_seq], dim=-1)) # [B, 7, H]
        deep_preds = self.forecast_head(final_out) # [B, 7, 3]

        # 6. Direct Autoregressive (AR) Residual Highway from Past 28-Day Sales (col 0) & Promo (col 1)
        past_sales = past_x[:, :, 0] # [B, 28]
        past_promo = past_x[:, :, 1] # [B, 28]
        ar_base = self.ar_sales_linear(past_sales).view(B, self.forecast_horizon, self.num_quantiles)
        ar_nonlin = self.ar_sales_grn(torch.cat([past_sales, past_promo], dim=-1)).view(B, self.forecast_horizon, self.num_quantiles)

        return deep_preds + 0.5 * ar_base + 0.5 * ar_nonlin


# Combined Huber + Quantile + Raw-Scale Loss for Joint MAE & RMSE Optimization
class CombinedHuberQuantileLoss(nn.Module):
    def __init__(self, mean, scale):
        super().__init__()
        self.huber_p50 = nn.HuberLoss(delta=1.35)
        self.quantiles = [0.1, 0.5, 0.9]
        self.mean = mean
        self.scale = scale

    def forward(self, preds, targets_scaled, targets_raw):
        # 1. Strong Huber loss on P50 (index 1) for crisp point accuracy
        p50 = preds[:, :, 1]
        loss_huber = self.huber_p50(p50, targets_scaled)

        # 2. Pinball loss on all 3 quantiles (P10, P50, P90)
        targets_exp = targets_scaled.unsqueeze(-1)
        q_losses = []
        for i, q in enumerate(self.quantiles):
            err = targets_exp - preds[:, :, i:i+1]
            q_losses.append(torch.max((q - 1.0) * err, q * err))
        loss_quantile = torch.mean(torch.cat(q_losses, dim=-1))

        # 3. Mild raw-space L1+L2 penalty to prevent exponential peak explosions
        unscaled_log = p50 * self.scale + self.mean
        preds_raw = torch.expm1(torch.clamp(unscaled_log, min=-5.0, max=14.0))
        denom = torch.mean(targets_raw) + 1.0
        raw_mae = torch.mean(torch.abs(preds_raw - targets_raw)) / denom
        raw_rmse = torch.sqrt(torch.mean((preds_raw - targets_raw) ** 2) + 1e-6) / denom

        return 0.50 * loss_huber + 0.35 * loss_quantile + 0.15 * (0.6 * raw_mae + 0.4 * raw_rmse)


# Instantiate Enhanced TFT & Train for 12 Epochs with Top-K Checkpoint Averaging
enh_model = EnhancedTemporalFusionTransformer(
    static_cardinalities=cat_cardinalities,
    hidden_size=96,
    embedding_size=24,
    num_heads=4,
    dropout=0.20
).to(DEVICE)

enh_criterion = CombinedHuberQuantileLoss(SCALER_MEAN, SCALER_SCALE)
print(f"Enhanced TFT Parameters: {sum(p.numel() for p in enh_model.parameters() if p.requires_grad):,}")

ENH_EPOCHS = 12
enh_optimizer = torch.optim.AdamW(enh_model.parameters(), lr=1e-3, weight_decay=3e-4)
enh_scheduler = torch.optim.lr_scheduler.OneCycleLR(
    enh_optimizer, max_lr=2e-3, epochs=ENH_EPOCHS, steps_per_epoch=len(train_loader), pct_start=0.15
)
enh_scaler = torch.amp.GradScaler('cuda')

top_checkpoints = [] # Stores (score, state_dict) of top 3 epochs

print("=" * 75)
print(f"TRAINING ENHANCED 35-DAY ATTENTION TFT ({ENH_EPOCHS} EPOCHS)")
print("=" * 75)

for epoch in range(1, ENH_EPOCHS + 1):
    t0 = time.time()
    enh_model.train()
    tr_loss = 0.0
    for batch in train_loader:
        px = batch["past_x"].to(DEVICE, non_blocking=True)
        fx = batch["future_x"].to(DEVICE, non_blocking=True)
        sx = batch["static_x"].to(DEVICE, non_blocking=True)
        ty = batch["y"].to(DEVICE, non_blocking=True)
        yr = batch["y_raw"].to(DEVICE, non_blocking=True)

        enh_optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda'):
            out = enh_model(px, fx, sx)
            loss = enh_criterion(out, ty, yr)

        enh_scaler.scale(loss).backward()
        enh_scaler.unscale_(enh_optimizer)
        torch.nn.utils.clip_grad_norm_(enh_model.parameters(), max_norm=1.0)
        enh_scaler.step(enh_optimizer)
        enh_scaler.update()
        enh_scheduler.step()
        tr_loss += loss.item()

    tr_loss /= len(train_loader)
    val_metrics, _, _ = evaluate_validation(enh_model, val_loader)
    elapsed = time.time() - t0

    # Composite validation score combining WAPE and RMSE
    comp_score = val_metrics["MAE"] + 0.35 * val_metrics["RMSE"]
    print(f"Epoch [{epoch:02d}/{ENH_EPOCHS:02d}] | Train Loss: {tr_loss:.4f} | "
          f"Val MAE: {val_metrics['MAE']:6.3f} | Val RMSE: {val_metrics['RMSE']:6.3f} | "
          f"Val WAPE: {val_metrics['WAPE (%)']:5.2f}% | R²: {val_metrics['R2']:.4f} | Time: {elapsed:4.1f}s")

    # Keep Top-3 best checkpoints for Stochastic Weight Averaging (SWA)
    cpu_state = {k: v.cpu().clone() for k, v in enh_model.state_dict().items()}
    top_checkpoints.append((comp_score, epoch, cpu_state))
    top_checkpoints.sort(key=lambda x: x[0])
    top_checkpoints = top_checkpoints[:3]

print("\nTop 3 Epochs Selected for Weight Averaging:", [ep for _, ep, _ in top_checkpoints])

# Average the weights of the Top-3 checkpoints (SWA)
avg_state = copy.deepcopy(top_checkpoints[0][2])
for k in avg_state.keys():
    if avg_state[k].is_floating_point():
        avg_state[k] = sum(ckpt[2][k] for ckpt in top_checkpoints) / len(top_checkpoints)

torch.save(avg_state, "tft_enhanced_swa_champion.pth")
enh_model.load_state_dict(avg_state)
print("Saved Top-3 Weight-Averaged Champion to 'tft_enhanced_swa_champion.pth'!")

Enhanced TFT Parameters: 1,957,028
TRAINING ENHANCED 35-DAY ATTENTION TFT (12 EPOCHS)
Epoch [01/12] | Train Loss: 0.1810 | Val MAE: 13.036 | Val RMSE: 41.635 | Val WAPE: 30.53% | R²: 0.6469 | Time: 153.6s
Epoch [02/12] | Train Loss: 0.1355 | Val MAE: 12.495 | Val RMSE: 35.741 | Val WAPE: 29.26% | R²: 0.7398 | Time: 147.4s
Epoch [03/12] | Train Loss: 0.1303 | Val MAE: 12.267 | Val RMSE: 36.142 | Val WAPE: 28.73% | R²: 0.7339 | Time: 147.6s
Epoch [04/12] | Train Loss: 0.1274 | Val MAE: 12.063 | Val RMSE: 34.766 | Val WAPE: 28.25% | R²: 0.7538 | Time: 147.8s
Epoch [05/12] | Train Loss: 0.1252 | Val MAE: 12.094 | Val RMSE: 35.026 | Val WAPE: 28.33% | R²: 0.7501 | Time: 147.0s
Epoch [06/12] | Train Loss: 0.1234 | Val MAE: 12.491 | Val RMSE: 37.018 | Val WAPE: 29.25% | R²: 0.7209 | Time: 146.8s
Epoch [07/12] | Train Loss: 0.1211 | Val MAE: 12.015 | Val RMSE: 34.732 | Val WAPE: 28.14% | R²: 0.7543 | Time: 147.5s
Epoch [08/12] | Train Loss: 0.1185 | Val MAE: 12.132 | Val RMSE: 35.047 | Val WAP

In [11]:
# =====================================================================
# Download Trained Model Weights & Forecast CSV from Google Colab
# =====================================================================
import os
from google.colab import files

# 1. Download the 3.1M-parameter upgraded TFT model from Cell 6
if os.path.exists("tft_upgraded_champion.pth"):
    print("Downloading tft_upgraded_champion.pth...")
    files.download("tft_upgraded_champion.pth")

# 2. Download the Enhanced SWA TFT model (if you ran Cell 9)
if os.path.exists("tft_enhanced_swa_champion.pth"):
    print("Downloading tft_enhanced_swa_champion.pth...")
    files.download("tft_enhanced_swa_champion.pth")

# 3. Download the exported test predictions CSV from Cell 7
if os.path.exists("tft_test_forecasts_upgraded.csv"):
    print("Downloading tft_test_forecasts_upgraded.csv...")
    files.download("tft_test_forecasts_upgraded.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>